<a href="https://colab.research.google.com/github/bauyrzhantorebek-droid/deep-learning-final-project/blob/bauyrzhantorebek-droid-patch-1/notebooks/02_distilbert_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install transformers -q

import pandas as pd
import torch
from torch.utils.data import DataLoader
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from torch.optim import AdamW  # Исправленный импорт
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# Загружаем датасет
class ToxicCommentDataset(torch.utils.data.Dataset):
    def __init__(self, texts, targets, tokenizer, max_len):
        self.texts = texts
        self.targets = targets
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])

        # Используем современный прямой вызов токенизатора вместо encode_plus
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'targets': torch.tensor(self.targets[item], dtype=torch.float)
        }

print("Загрузка данных...")
df = pd.read_csv('train.csv')
# Для скорости возьмем 10% данных
df = df.sample(frac=0.1, random_state=42).reset_index(drop=True)

X = df['comment_text'].values
y = df[['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']].values

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Настройка DistilBERT
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используем устройство: {device}")

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=6, problem_type="multi_label_classification")
model.to(device)

train_dataset = ToxicCommentDataset(X_train, y_train, tokenizer, max_len=128)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

# Используем обновленный AdamW
optimizer = AdamW(model.parameters(), lr=2e-5)

# Тренировочный цикл (1 эпоха)
print("Начинаем обучение...")
model.train()
for batch in tqdm(train_loader):
    optimizer.zero_grad()
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    targets = batch['targets'].to(device)

    outputs = model(input_ids, attention_mask=attention_mask, labels=targets)
    loss = outputs.loss
    loss.backward()
    optimizer.step()

print("Обучение завершено! Модель готова к валидации.")

Загрузка данных...
Используем устройство: cuda


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Начинаем обучение...


100%|██████████| 798/798 [02:18<00:00,  5.78it/s]

Обучение завершено! Модель готова к валидации.
